In [ ]:
from numpy.ma.extras import setdiff1d
!pip install geopandas matplotlib
!pip install geodatasets

In [ ]:
import pandas as pd
import os
import geopandas as gpd
import geodatasets
from shapely.geometry import Point
from geopandas import GeoSeries, GeoDataFrame
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.patches import Patch
import seaborn as sns
import seaborn.objects as so
import numpy as np
import gc
from datetime import datetime, timezone
import matplotlib.dates as mdates
from scipy import stats
import pyproj

# Load in CSVs

"C:\Users\25298423\OneDrive - National University of Ireland, Galway\Desktop\Data_Downloads\RockyShoreMacroalgae_unzipped"

In [ ]:
os.listdir("/Users/25298423/OneDrive - National University of Ireland, Galway/Desktop/Data_Downloads/RockyShoreMacroalgae_unzipped")

In [ ]:
rocky_algae = pd.read_csv("/Users/25298423/OneDrive - National University of Ireland, Galway/Desktop/Data_Downloads/RockyShoreMacroalgae_unzipped/RockyShoreMacroalgae.csv")
rocky_algae

In [ ]:
os.listdir("/Users/25298423/OneDrive - National University of Ireland, Galway/Desktop/Data_Downloads/SeaweedsOfIreland_unzipped")


In [ ]:
seaweed = pd.read_csv("/Users/25298423/OneDrive - National University of Ireland, Galway/Desktop/Data_Downloads/SeaweedsOfIreland_unzipped/SeaweedsOfIreland.csv")
seaweed

# rocky_algae EDA

In [ ]:
rocky_algae.columns

In [ ]:
rocky_algae.describe()

In [ ]:
rocky_algae.drop(columns=['SurveyKey','SampleKey','GridReference','SiteKey','ImagePath'], inplace=True)
rocky_algae

In [ ]:
rocky_algae.dtypes

In [ ]:
rocky_algae['StartDate']

In [ ]:
import re

In [ ]:
rocky_algae['StartDate'] = [re.findall(r"\d{1,2}\/\d{1,2}\/\d{4}", x)[0] for x in rocky_algae['StartDate']]

In [ ]:
rocky_algae['EndDate'] = [re.findall(r"\d{1,2}\/\d{1,2}\/\d{4}", x)[0] for x in rocky_algae['EndDate']]

In [ ]:
rocky_algae['Date'] = [re.findall(r"\d{1,2}\/\d{1,2}\/\d{4}", x)[0] for x in rocky_algae['Date']]

In [ ]:
rocky_algae

In [ ]:
(rocky_algae['StartDate'] != rocky_algae['EndDate']).sum()


In [ ]:
## start date is always equal to end date, can eliminate those rows
rocky_algae.drop(columns=['StartDate','EndDate'], inplace=True)
rocky_algae

In [ ]:
rocky_algae['Date'] = pd.to_datetime(rocky_algae.Date, dayfirst =True)
rocky_algae.dtypes

In [ ]:
len(rocky_algae.TaxonName.unique())
## 223 unique rocky_algae

In [ ]:
sns.scatterplot( rocky_algae, x= 'East', y='North', hue='TaxonName')
plt.show()

In [ ]:
taxon_counts = rocky_algae['TaxonName'].value_counts().reset_index()
taxon_counts.columns = ['TaxonName','Count']
taxon_counts

In [ ]:
rocky_fucus = rocky_algae[rocky_algae.TaxonName.str.contains('Fucus')]

In [ ]:
sns.scatterplot( rocky_fucus, x= 'East', y='North', hue='TaxonName')
plt.show()

In [ ]:
## try to get this onto a map of Ireland using geopandas

url = "https://geodata.ucdavis.edu/gadm/gadm4.1/shp/gadm41_IRL_shp.zip"
uk_url = "https://geodata.ucdavis.edu/gadm/gadm4.1/shp/gadm41_GBR_shp.zip"

ireland = gpd.read_file(url, layer= "gadm41_IRL_0")

uk = gpd.read_file(uk_url, layer="gadm41_GBR_0")
uk_regions = gpd.read_file(uk_url, layer="gadm41_GBR_1")
northern_ireland = uk_regions[uk_regions["NAME_1"]=="Northern Ireland"]

In [ ]:
## EPSG: 4326 will buffer in decimal degrees, not meters. For ease, we want to load it in as 4326, and then reproject it to EPSG: 2157 for Irish Transverse Mercator
island_4326 = gpd.GeoDataFrame(pd.concat([ireland,northern_ireland], ignore_index=True), crs="EPSG:4326")

In [ ]:
island = island_4326.to_crs("EPSG:2157")

In [ ]:
## need to get rocky_algae into geodataframe in order to plot on map, use 4326 first
rocky_algae = gpd.GeoDataFrame(
    rocky_algae,
    geometry = gpd.points_from_xy(rocky_algae['East'], rocky_algae['North']),
crs="EPSG:4326")
## then reproject to ITM (2157)
rocky_algae = rocky_algae.to_crs('EPSG:2157')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))

island.plot(ax=ax, color="lightgray", edgecolor="black")
rocky_algae.plot(ax=ax, color="blue", markersize=20)

plt.show()


In [ ]:
rocky_algae

In [ ]:
## need to redefine rocky_fucus so that it takes on the geopoints
rocky_fucus = rocky_algae[rocky_algae.TaxonName.str.contains('Fucus')]
rocky_fucus

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))

island.plot(ax=ax, color="lightgray", edgecolor="black")
rocky_fucus.plot(ax=ax, color="blue", markersize=20)

plt.show()

In [ ]:
## want to differentiate between different algaes - take first name of their Taxon Name to get categories

rocky_algae.TaxonName.sort_values()

In [ ]:
rocky_algae["Genus"] = rocky_algae["TaxonName"].str.extract(r'^(\w+)')
rocky_algae["Genus"]

In [ ]:
len(rocky_algae.Genus.unique())
## 128 unique Genus

In [ ]:
rocky_genus = rocky_algae.Genus.unique()


In [ ]:
rocky_genus_df = pd.DataFrame(data=rocky_genus, columns=["Genus"])

In [ ]:
# rocky_genus_df.to_csv('rocky_genus.csv', index=False)
## only worked the first time ran cell, because now there is an updated version that was edited outside of python

In [ ]:
rocky_algae[rocky_algae.TaxonName.str.contains('Lichina')]

In [ ]:
rocky_orders = pd.read_csv("/Users/25298423/PycharmProjects/JupyterProject1/rocky_genus.csv")


rocky_orders = rocky_orders.fillna('Other')
rocky_orders

In [ ]:
rocky_algae = rocky_algae.join(rocky_orders.set_index('Genus'), on='Genus')

In [ ]:
## need to redefine rocky_fucus so that it takes on the genus, type, and order

rocky_fucus = rocky_algae[rocky_algae.TaxonName.str.contains('Fucus')]
## Make another one for the whole order

rocky_fucales = rocky_algae[rocky_algae.Order.str.contains('Fucales')]
rocky_laminariales = rocky_algae[rocky_algae.Order.str.contains('Laminariales')]

In [ ]:
rocky_fucales

In [ ]:
rocky_others = rocky_algae[rocky_algae.Order.str.contains('Other|Corallinales')]
rocky_others

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))

island.plot(ax=ax, color="lightgray", edgecolor="black")

rocky_laminariales.plot(ax=ax, color="red",marker = 'o', markersize=20)
rocky_fucales.plot(ax=ax, color="blue", marker = '^', markersize=10)
rocky_others.plot(ax=ax, color="green", marker = '*', markersize=5)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))

island.plot(ax=ax, color="lightgray", edgecolor="black")

rocky_laminariales.plot(ax=ax, color="red",marker = 'o', markersize=20)
rocky_fucales.plot(ax=ax, color="blue", marker = '^', markersize=10)

ax.set_xlim(-10,-8.75)
ax.set_ylim(53,53.5)
plt.show()

In [ ]:
rocky_laminariales.TaxonName.sort_values().unique()

In [ ]:
rocky_fucales.TaxonName.sort_values().unique()


# seaweed EDA

In [ ]:
seaweed

In [ ]:
seaweed.columns

In [ ]:
seaweed.describe()

In [ ]:
seaweed.dtypes

In [ ]:
seaweed.drop(columns=['SurveyKey','SampleKey','SiteKey','ImagePath'], inplace=True)

In [ ]:
seaweed

In [ ]:
seaweed.ZeroAbundance.sum()

In [ ]:
seaweed.UnderValidation.sum()

In [ ]:
len(seaweed[~seaweed.Determiner.str.contains('Not specified')])
## used negative lookahead (~) to get the opposite of "contains"

In [ ]:
print(len(seaweed[~seaweed.SiteName.str.contains('Not specified')]))
seaweed[~seaweed.SiteName.str.contains('Not specified')].head()

In [ ]:
seaweed.drop(columns= ['Determiner','ZeroAbundance','UnderValidation'], inplace=True)

In [ ]:
seaweed.Date.unique()

In [ ]:
seaweed[['StartDate','EndDate','Date']]

In [ ]:
seaweed.StartDate.unique()

In [ ]:
seaweed.StartDate = pd.to_datetime(seaweed.StartDate, dayfirst=True).dt.normalize()
seaweed.EndDate = pd.to_datetime(seaweed.EndDate, dayfirst=True).dt.normalize()

In [ ]:
seaweed.dtypes

In [ ]:
seaweed['Duration'] = seaweed.EndDate - seaweed.StartDate

In [ ]:
seaweed.Duration.unique()

In [ ]:
seaweed.DateType.unique()

In [ ]:
seaweed[seaweed.Duration.isin(['29 days','30 days'])]['DateType'].unique()

## okay so DateType refers to if the survey was over the course of a day, a month, or a year

In [ ]:
seaweed.Projection.unique()

In [ ]:
seaweed.head()

In [ ]:
seaweed.GridReference.unique()

In [ ]:
## BIG PROBLEM, Eastings and Northings are obnoxious and do not work with any of my other data. Need to convert to lat/long.
## LUCKY - found code at https://webdistortion.com/articles/irish-grid-references-in-python that has already defined a function to convert East/North to lat and long. godbless Paul Anthony of Belfast.


crs_osi = "EPSG:29902"
crs_wgs84 = "EPSG:4326"

transformer = pyproj.Transformer.from_crs(crs_osi, crs_wgs84, always_xy = True)

def ingrs_to_latlng(easting, northing):

    lng, lat = transformer.transform(easting, northing)

    return lat, lng

## had to update it a bit with new pyproj syntax

In [ ]:
coordinates = seaweed.apply(lambda row: ingrs_to_latlng(row['East'],row['North']), axis =1)

In [ ]:
ingrs_to_latlng(326500,261500)

In [ ]:
seaweed[['Latitude', 'Longitude']] = pd.DataFrame(coordinates.tolist(), index=seaweed.index)

In [ ]:
seaweed.columns

In [ ]:
seaweed = seaweed[['StartDate','EndDate','Duration','DateType','Latitude','Longitude','Precision','TaxonName','SiteName','Vice-county','Recorder','Source']]

In [ ]:
seaweed

In [ ]:
seaweed.groupby('DateType').count()

In [ ]:
seaweed["Genus"] = seaweed["TaxonName"].str.extract(r'^(\w+)')
seaweed["Genus"]

In [ ]:
len(seaweed.Genus.unique())

In [ ]:
seaweed.Genus.unique()

In [ ]:
set(["apple", "mango", "banana"]) - set(["mango", "blueberry", "strawberry"])

In [ ]:
seaweed_notrocky_genus = list(set(seaweed.Genus.unique()) - set(rocky_algae.Genus.unique()))

In [ ]:
set(rocky_algae.Genus.unique()) - set(seaweed.Genus.unique())

In [ ]:
seaweed_notrocky_genus_df = pd.DataFrame(data=seaweed_notrocky_genus, columns=["Genus"])
#seaweed_notrocky_genus_df.to_csv('seaweed_notrocky_genus.csv', index=False)

In [ ]:
seaweed[seaweed.TaxonName.str.contains("Carpomitra")].head()

In [ ]:
seaweed_orders = pd.read_csv("/Users/25298423/PycharmProjects/JupyterProject1/seaweed_notrocky_genus.csv")

In [ ]:
seaweed_orders.fillna('Other', inplace=True)

In [ ]:
seaweed_orders

In [ ]:
rocky_orders

In [ ]:
orders = pd.concat([rocky_orders, seaweed_orders]).sort_values("Genus")
orders
## orders (df) contains all the Genus present in both dataframes, their type, and the order they belong to (Laminariales, Fucales, Corallinales, or Other)

In [ ]:
seaweed = seaweed.join(orders.set_index('Genus'), on='Genus')
## need to redefine seaweed so that it takes on the genus, type, and order
seaweed

In [ ]:
## need to get seaweed into geodataframe in order to plot on map
seaweed = gpd.GeoDataFrame(
    seaweed,
    geometry = gpd.points_from_xy(seaweed['Longitude'], seaweed['Latitude']),
crs="EPSG:4326")
## convert to ITM (2157
seaweed = seaweed.to_crs('EPSG:2157')

In [ ]:
## need to redefine seaweed so that it takes on the genus, type, and order

seaweed_fucus = seaweed[seaweed.TaxonName.str.contains('Fucus')]
## Make another one for the whole order

seaweed_fucales = seaweed[seaweed.Order.str.contains('Fucales')]
seaweed_laminariales = seaweed[seaweed.Order.str.contains('Laminariales')]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))

island.plot(ax=ax, color="lightgray", edgecolor="black")

seaweed_laminariales.plot(ax=ax, color="red",marker = 'o', markersize=20)
seaweed_fucales.plot(ax=ax, color="blue", marker = '^', markersize=10)

plt.show()

In [ ]:
sns.histplot(data=seaweed[seaweed['Order'].str.contains(r'Laminariales|Fucales')], x='StartDate', hue='Order')

In [ ]:
seaweed.groupby(['Latitude','Longitude','StartDate','EndDate']).count()

# Now put the dataframes together

In [ ]:
rocky_algae

In [ ]:
rocky_algae['StartDate'] = rocky_algae['Date']
rocky_algae['EndDate'] = rocky_algae['Date']
rocky_algae.drop(columns=['Date','TaxonVersionKey','ZeroAbundance','UnderValidation','RecordKey'], inplace=True)

In [ ]:
rocky_algae.rename(columns={'East':'Longitude','North':'Latitude'}, inplace=True)

In [ ]:
rocky_duration = rocky_algae['EndDate'] - rocky_algae['StartDate']
rocky_duration

In [ ]:
rocky_algae['Duration'] = rocky_duration

In [ ]:
rocky_algae.rename(columns={'Determiner':'Source'}, inplace=True)

In [ ]:
rocky_algae['Original_Data'] = ['RockyShoreMacroalgae, EPA' for x in range(rocky_algae.shape[0])]
seaweed['Original_Data'] = ['SeaweedsOfIreland, BPS' for x in range(seaweed.shape[0])]

In [ ]:
rocky_algae = rocky_algae[['StartDate','EndDate', 'Duration','DateType','Latitude','Longitude','Precision','geometry','SiteName','TaxonName','Genus', 'Type','Order','Recorder','Source','Original_Data']]
rocky_algae.head()

In [ ]:
seaweed = seaweed[['StartDate','EndDate','Duration','DateType','Latitude','Longitude','Precision','geometry','SiteName','TaxonName','Genus','Type','Order','Recorder','Source','Original_Data']]

In [ ]:
rocky_algae.head()


In [ ]:
seaweed.head()

In [ ]:
## all_algae is the direct combination of seaweed and rocky_algae
all_algae = pd.concat([seaweed, rocky_algae])

In [ ]:
all_algae.sort_values('StartDate', inplace=True)

In [ ]:
all_algae.reset_index(drop=True, inplace=True)

In [ ]:
all_algae

In [ ]:
all_algae.dtypes

In [ ]:
sns.histplot(data=all_algae[all_algae['Order'].str.contains(r'Laminariales|Fucales')], x='StartDate', hue='Order')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))

island.plot(ax=ax, color="lightgray", edgecolor="black")

all_algae[all_algae['Order'].str.contains('Laminariales')].plot(ax=ax, color="red",marker = 'o', markersize=20)
all_algae[all_algae['Order'].str.contains('Fucales')].plot(ax=ax, color="blue", marker = '^', markersize=10)
plt.title('Macroalgal Orders Laminariales and Fucales')
plt.show()

In [ ]:
all_algae.groupby('Precision').count()
## most are very precise, <6000 are basically estimates

In [ ]:
all_algae[all_algae.Precision == 10000].groupby('DateType').count()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))

island.plot(ax=ax, color="lightgray", edgecolor="black")

all_algae[(all_algae['Order'].str.contains('Laminariales')) & (all_algae['Precision'] == 100)].plot(ax=ax, color="red",marker = 'o', markersize=20)
all_algae[(all_algae['Order'].str.contains('Fucales')) & (all_algae['Precision'] == 100)].plot(ax=ax, color="blue", marker = '^', markersize=10)

plt.show()

In [ ]:
all_algae.to_csv('all_algae.csv', index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))



all_algae[(all_algae['Order'].str.contains('Laminariales')) & (all_algae['Precision'] == 100)].plot(ax=ax, color="red",marker = 'o', markersize=20)
all_algae[(all_algae['Order'].str.contains('Fucales')) & (all_algae['Precision'] == 100)].plot(ax=ax, color="blue", marker = '^', markersize=10)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))

all_algae[(all_algae['Order'].str.contains('Other')) & (all_algae['Precision'] == 100)].plot(ax=ax, color="lightgreen",marker = '^', markersize=20, label='Other')
all_algae[(all_algae['Order'].str.contains('Corallinales')) & (all_algae['Precision'] == 100)].plot(ax=ax, color="pink",marker = 'o', markersize=20, label='Corallinales')
all_algae[(all_algae['Order'].str.contains('Laminariales')) & (all_algae['Precision'] == 100)].plot(ax=ax, color="red",marker = 'o', markersize=20, label='Laminariales')
all_algae[(all_algae['Order'].str.contains('Fucales')) & (all_algae['Precision'] == 100)].plot(ax=ax, color="blue", marker = '^', markersize=10, label='Fucales')
ax.legend()
plt.gcf().savefig('all_algae_map.png', dpi=600)
plt.show()


In [ ]:
all_algae.head()

In [ ]:
all_algae.DateType.isna().sum()

In [ ]:
day_algae = all_algae[(all_algae.DateType.str.contains('D')) & (all_algae.Precision == 100)]

In [ ]:
day_algae_counts = day_algae.groupby(['Latitude','Longitude'])['StartDate'].nunique().reset_index()
day_algae_counts.head()

In [ ]:
day_algae_multiday_sites = day_algae_counts[day_algae_counts.StartDate > 1]
day_algae_multiday_sites.sort_values(by=['StartDate'],ascending=False, inplace=True)
day_algae_multiday_sites


In [ ]:
multi_algae = day_algae.merge(day_algae_multiday_sites[['Latitude','Longitude']], how='inner', on=['Latitude','Longitude'])

In [ ]:
multi_algae.head()

In [ ]:
multi_algae.DateType.unique()

In [ ]:
multi_algae.sort_values(['Latitude', 'Longitude'], inplace=True)

In [ ]:
multi_algae.Precision.unique()

In [ ]:
multi_algae



In [ ]:
multi_algae.sort_values('StartDate')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))

island.plot(ax=ax, color="lightgray", edgecolor="black")

all_algae[all_algae['Order'].str.contains('Other')].plot(ax=ax, color="lightgreen",marker = '*', markersize=20, label='Other')
all_algae[all_algae['Order'].str.contains('Corallinales')].plot(ax=ax, color="pink",marker = 's', markersize=20, label='Corallinales')

all_algae[all_algae['Order'].str.contains('Laminariales')].plot(ax=ax, color="red",marker = 'o', markersize=20, label='Laminariales')
all_algae[all_algae['Order'].str.contains('Fucales')].plot(ax=ax, color="blue", marker = '^', markersize=10, label='Fucales')

plt.title('All Macroalgae Surveys')

plt.savefig('all_algae_map.png', dpi=600)
plt.show()

In [ ]:
multi_algae.to_csv('multi_algae.csv', index=False)

In [ ]:
multi_algae.sort_values('StartDate')

In [ ]:
day_site_group = multi_algae[['StartDate','Latitude','Longitude']].drop_duplicates()
day_site_group.sort_values('StartDate', ascending=False)

In [ ]:
plt.hist(data=day_site_group, x='StartDate', bins=150)

In [ ]:
day_site_group.dtypes

In [ ]:
print(day_site_group['StartDate'].value_counts().head(10))

In [ ]:
day_site_group[day_site_group['StartDate']=='1999-12-31']

In [ ]:
day_site_group[day_site_group['StartDate']=='1983-07-20']

In [ ]:
## at this point I have realized that I need to remove all the surveys that were inputed on 12-31 or 01-01, as they are not the real dates of the survey

In [ ]:
df = multi_algae.copy()

In [ ]:
df['Month'] = df.StartDate.dt.month
df['Day'] = df.StartDate.dt.day
df['Year'] = df.StartDate.dt.year

In [ ]:
fake_days = ((df['Month'] == 12) & (df['Day'] == 31)) |  ((df['Month'] == 1) & (df['Day'] == 1))

In [ ]:
multi_algae_real = multi_algae[~fake_days]
multi_algae_real

In [ ]:
# Check which sites are in the 'bad' dates
multi_algae_real.StartDate.value_counts().head(10)

In [ ]:
multi_algae_real = multi_algae_real[multi_algae_real.StartDate != '1983-7-20']

In [ ]:
multi_algae_real.sort_values('StartDate')


In [ ]:
multi_algae_real.StartDate.value_counts().head(10)

In [ ]:
day_site_group = multi_algae_real[['StartDate', 'Latitude', 'Longitude']].drop_duplicates()
day_site_group.sort_values('StartDate', ascending=False)
plt.hist(data=day_site_group, x='StartDate', bins=150)
plt.show()

In [ ]:
yearly_bins = pd.date_range(start=day_site_group['StartDate'].min(),
                             end=day_site_group['StartDate'].max(),
                             freq='YS') # 'MS' = Month Start

plt.figure(figsize=(12, 6))

# 2. Pass the generated dates into the 'bins' argument
plt.hist(day_site_group['StartDate'], bins=yearly_bins, color='seagreen', edgecolor='white')

# 3. Format the X-axis to show years and gridlines
plt.gca().xaxis.set_major_locator(mdates.YearLocator())
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.xticks(rotation=45)
plt.grid(axis='x', linestyle='--', alpha=0.5)

plt.title('Distribution of Seaweed Sampling')
plt.show()

In [ ]:
yearly_bins = pd.date_range(start=day_site_group['StartDate'].min(),
                             end=day_site_group['StartDate'].max(),
                             freq='YS') # 'MS' = Month Start

plt.figure(figsize=(12, 6))

plt.hist(day_site_group['StartDate'], bins=yearly_bins, color='seagreen', edgecolor='white')

plt.xlabel('Year')
plt.ylabel('Survey Count')
plt.title('Distribution of Macroalgae Surveying')
plt.savefig('algae_survey_histogram.png', dpi=600)
plt.show()

In [ ]:
multi_algae_real.to_csv('multi_algae_real.csv', index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))

island.plot(ax=ax, color="lightgray", edgecolor="black")



multi_algae_real[multi_algae_real['Order'].str.contains('Other')].plot(ax=ax, color="lightgreen",marker = '*', markersize=20, label='Other')
multi_algae_real[multi_algae_real['Order'].str.contains('Corallinales')].plot(ax=ax, color="pink",marker = 's', markersize=20, label='Corallinales')
multi_algae_real[multi_algae_real['Order'].str.contains('Laminariales')].plot(ax=ax, color="red",marker = 'o', markersize=20, label = 'Laminariales')
multi_algae_real[multi_algae_real['Order'].str.contains('Fucales')].plot(ax=ax, color="blue", marker = '^', markersize=10, label='Fucales')
ax.legend()

plt.title('One-Day, Precise, and Repeated Macroalgae Surveys')

plt.savefig('multi_algae_real_map.png', dpi=600)
plt.show()

In [ ]:
multi_algae_real

In [ ]:
multi_algae_real[multi_algae_real["Order"]=='Corallinales']['TaxonName'].sort_values().unique()

In [ ]:
## realized at this point, that not all of these species are Order = Corallinales
## These species are Order = Hapalidiales: Lithothamnion sonderi, ~Lithothamnion corallioides~, Lithothamnion glaciale, ~Phymatolithon calcareum~, Phymatolithon purpureum, Phymatolithon lenormandii, Mesophyllum lichenoides


##additionally, "maerl" is only free-living coralline algae, and does not include Crustose Corraline Algae (CCA), which is what the majority of the order Corallinales contains


In [ ]:
multi_algae_real['TaxonName'].sort_values().unique()

In [ ]:
multi_algae_real['Genus'].sort_values().unique()

In [ ]:
genus_order = pd.read_excel("/Users/25298423/OneDrive - National University of Ireland, Galway/Desktop/Data_Downloads/algae_genus_order.xlsx")

In [ ]:
genus_order

In [ ]:
genus_order['Genus'] = genus_order['Genus'].str.strip().str.capitalize()

In [ ]:
multi_algae_real.loc[:,'Genus'] = multi_algae_real['Genus'].str.strip().str.capitalize()

In [ ]:
multi_algae_real = multi_algae_real.drop(columns='Order')

In [ ]:
multi_algae_real = pd.merge(multi_algae_real, genus_order, how='left', on='Genus')

In [ ]:
multi_algae_real

In [ ]:
multi_algae_real[multi_algae_real['Order']=='Hapalidiales']['TaxonName'].sort_values().unique()

## matches the list of what I messed up! yay!

In [ ]:
genus_funcgroup = pd.read_excel("/Users/25298423/OneDrive - National University of Ireland, Galway/Desktop/Data_Downloads/algae_genus_funcgroup_final.xlsx")
genus_funcgroup.head()

In [ ]:
multi_algae_real = pd.merge(multi_algae_real, genus_funcgroup, how='left', on='Genus')

In [ ]:
multi_algae_real

In [ ]:
## If i only have 1 day surveys are should remove one of the dates (but check to make sure they are all the same first

(multi_algae_real['StartDate'] != multi_algae_real['EndDate']).sum()

## can drop one of the columns

In [ ]:
multi_algae_real = multi_algae_real.drop(columns='EndDate')

In [ ]:
multi_algae_real.rename(columns={'StartDate':'Date'}, inplace=True)

In [ ]:
## can also now drop Duration, DateType, and Precision (They are all 1 day, Daily, and Precision =100 at this point)
multi_algae_real.head()

In [ ]:
multi_algae_real = multi_algae_real.drop(columns=['Duration','DateType', 'Precision'])

In [ ]:
multi_algae_real.columns

In [ ]:
multi_algae_real = multi_algae_real[['Date','TaxonName','Genus', 'Order','Type','Functional Group', 'Latitude', 'Longitude', 'geometry', 'SiteName', 'Recorder', 'Source', 'Original_Data']]

In [ ]:
multi_algae_real

In [ ]:
multi_algae_real[multi_algae_real['Functional Group'].isna()]

In [ ]:
multi_algae_real[multi_algae_real['TaxonName'] == 'Chlorophyta']

In [ ]:
taxon_bridge = {'Corallinaceae crusts': 'CCA',
                'Corallinophycidae':'CCA',
                'Chlorophyta': 'Opportunistic'}

for taxon, group in taxon_bridge.items():
    mask = (multi_algae_real['TaxonName'] == taxon) & (multi_algae_real['Functional Group'].isna())
    multi_algae_real.loc[mask, 'Functional Group'] = group

print(f"Remaining NaNs: {multi_algae_real['Functional Group'].isna().sum()}")

In [ ]:
multi_algae_real['Functional Group'].sort_values().unique()

In [ ]:
## I think everything is all set now! Plotting! (again)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))

island.plot(ax=ax, color="lightgray", edgecolor="black")


multi_algae_real[multi_algae_real['Functional Group'].str.contains('Kelp')].plot(ax=ax, color="g",marker = 's', markersize=20, label='Kelp')
multi_algae_real[multi_algae_real['Functional Group'].str.contains('Wrack')].plot(ax=ax, color="m",marker = '^', markersize=20, label='Wrack')
multi_algae_real[multi_algae_real['Functional Group'].str.contains('Maerl')].plot(ax=ax, color="#fa6686",marker = 'o', markersize=20, label='Maerl')
multi_algae_real[multi_algae_real['Functional Group'].str.contains('CCA')].plot(ax=ax, color="c",marker = '*', markersize=10, label='CCA')

plt.title('Historical Irish Macroalgae Surveys')
plt.legend()
plt.savefig('maerl_kelp_wrack.png', dpi=600)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

island.plot(ax=ax, color="lightgray", edgecolor="black")


multi_algae_real[multi_algae_real['Functional Group'].str.contains('Kelp')].plot(ax=ax, color="g",marker = 's', markersize=30, label='Kelp')
multi_algae_real[multi_algae_real['Functional Group'].str.contains('Wrack')].plot(ax=ax, color="m",marker = '^', markersize=30, label='Wrack')
multi_algae_real[multi_algae_real['Functional Group'].str.contains('Maerl')].plot(ax=ax, color="#fa6686",marker = 'o', markersize=30, label='Maerl')
multi_algae_real[multi_algae_real['Functional Group'].str.contains('CCA')].plot(ax=ax, color="c",marker = '*', markersize=20, label='CCA')

plt.xlim(450000,550000)
plt.ylim(700000,750000)
plt.title('Historical Macroalgae Surveys - Galway Bay')

plt.legend()
plt.tight_layout()
plt.savefig('macroalgae_galwaybay.png', dpi=600)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

island.plot(ax=ax, color="lightgray", edgecolor="black")


multi_algae_real[multi_algae_real['Functional Group'].str.contains('Kelp')].plot(ax=ax, color="g",marker = 's', markersize=30, label='Kelp')
multi_algae_real[multi_algae_real['Functional Group'].str.contains('Wrack')].plot(ax=ax, color="m",marker = '^', markersize=30, label='Wrack')
multi_algae_real[multi_algae_real['Functional Group'].str.contains('Maerl')].plot(ax=ax, color="#fa6686",marker = 'o', markersize=30, label='Maerl')
multi_algae_real[multi_algae_real['Functional Group'].str.contains('CCA')].plot(ax=ax, color="c",marker = '*', markersize=20, label='CCA')

plt.xlim(470000,550000)
plt.ylim(700000,735000)
plt.title('Historical Macroalgae Surveys - Connemara')

plt.legend()
plt.tight_layout()
plt.savefig('macroalgae_connemara.png', dpi=600)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

island.plot(ax=ax, color="#f5bff3", edgecolor="black")


multi_algae_real[multi_algae_real['Functional Group'].str.contains('Kelp')].plot(ax=ax, color="g",marker = 's', markersize=30, label='Kelp')
multi_algae_real[multi_algae_real['Functional Group'].str.contains('Wrack')].plot(ax=ax, color="m",marker = '^', markersize=30, label='Wrack')
multi_algae_real[multi_algae_real['Functional Group'].str.contains('Maerl')].plot(ax=ax, color="#fa6686",marker = 'o', markersize=30, label='Maerl')
multi_algae_real[multi_algae_real['Functional Group'].str.contains('CCA')].plot(ax=ax, color="c",marker = 'd', markersize=20, label='CCA')

plt.xlim(470000,550000)
plt.ylim(700000,735000)
plt.title('Historical Macroalgae Surveys - Connemara')

plt.legend()
plt.tight_layout()

plt.show()

## Folium Plots (Interactive Maps)

In [ ]:
import folium
from folium import plugins
from folium.plugins import MarkerCluster

# 1. Create a base map centered on your Connemara coordinates
# We use 'cartodbpositron' for a clean, gray background similar to your static map
m = folium.Map(location=[53.35, -9.85], zoom_start=10, tiles='cartodbpositron')

# 2. Define your styles to match your previous plot
shape_map = {
    'Kelp':  {'color': 'green',   'icon': 'square'},
    'Wrack': {'color': 'purple',  'icon': 'caret-up'},
    'Maerl': {'color': 'pink', 'icon': 'circle'},
    'CCA':   {'color': 'cadetblue', 'icon': 'star'}
}

marker_cluster = MarkerCluster(spiderfyOnMaxZoom=True).add_to(m)

## TEMPORARY ESPG 4325 map
map_data_4326 = multi_algae_real.to_crs(epsg=4326)

for group, style in shape_map.items():
    # Filter data
    subset = map_data_4326[map_data_4326['Functional Group'].str.contains(group, na=False)]

    for _, row in subset.iterrows():
        lat, lon = row.geometry.y, row.geometry.x

        folium.Marker(
            location=[lat, lon],
            icon=folium.Icon(color=style['color'], icon=style['icon'], prefix='fa'),
            tooltip=(f"<b>Group:</b> {group}<br>"
                     f"<b>Genus:</b> {row['Genus']}<br>"
                     f"<b>Lat:</b> {lat:.5f}<br>"
                     f"<b>Lon:</b> {lon:.5f}<br>"
                     f"<b>Site:</b> {row['SiteName']}<br>")
        ).add_to(marker_cluster)


# 4. Add a layer control so you can turn Kelp/Maerl on and off
folium.LayerControl().add_to(m)

# Save to an HTML file you can open in any browser
m.save('interactive_connemara_map.html')
m

In [ ]:
multi_algae_real.head()

## Buffer Locations
New problem: sites that are right next to each other are labeled under different lat/lon and site names. Need to use geopandas buffer to combine them into one site (but I don't want to lose their original lat/lon either)

In [ ]:
## New problem: sites that are right next to each other are labeled under different lat/lon and site names. Need to use geopandas buffer to combine them into one site (but I don't want to lose their original lat/lon either)
## and after trialnerror, don't want to use Site Name as the method for grouping, as about 2,000 of them are "Not specified"

# 1. make a copy with shorter name so I don't fuck it all up
algae_itm = multi_algae_real.copy()

# 2. make 50m buffers around every single point
buffers = algae_itm.geometry.buffer(50)

# 3. Dissolve the buffers so that the overlapping circles are in one multipolygon per area
dis_buf = gpd.GeoDataFrame(geometry=buffers, crs=algae_itm.crs).dissolve().reset_index()

# 4. explode the multipolygons back into individual "site polygons"
site_poly = dis_buf.explode(index_parts=True).reset_index(drop=True)
site_poly['cluster_id'] = site_poly.index

# 5. Spatial join the og points to the "site polys"
site_poly = gpd.GeoDataFrame(site_poly[['cluster_id', 'geometry']], crs=algae_itm.crs)
algae_itm = algae_itm.set_geometry('geometry')

algae_snap = gpd.sjoin(algae_itm,
                       site_poly,
                       how='left',
                       predicate='within')

# 6. Calculate the average center of all points in the blob to get new lat long
poly_center = algae_snap.groupby('cluster_id')['geometry'].apply(
    lambda g: Point(g.x.mean(), g.y.mean()))

algae_snap['snapped_geometry'] = algae_snap['cluster_id'].map(poly_center)

In [ ]:
algae_snap

In [ ]:
algae_snap['snapped_geometry'].crs = "EPSG:2157"

temp_4326 = algae_snap.set_geometry('snapped_geometry').to_crs(epsg=4326)
algae_snap['snap_lat'] = temp_4326.geometry.y
algae_snap['snap_lon'] = temp_4326.geometry.x


In [ ]:
algae_snap

In [ ]:
def get_best_name(group):
    # Filter out 'Not specified' and NaNs to find the "real" names
    real_names = group[group != 'Not specified'].dropna()

    if not real_names.empty:
        # Return the most frequent real name in this 50m bubble
        return real_names.mode()[0]

    return 'Unlabeled_Cluster' # If the whole 50m area has no names at all

# Create the new column
algae_snap['Standard_Site_Name'] = algae_snap.groupby('cluster_id')['SiteName'].transform(get_best_name)

In [ ]:
# 1. Create a base map centered on your Connemara coordinates
# We use 'cartodbpositron' for a clean, gray background similar to your static map
m = folium.Map(location=[53.35, -9.85], zoom_start=10, tiles='cartodbpositron')

# 2. Define your styles to match your previous plot
shape_map = {
    'Kelp':  {'color': 'green',   'icon': 'square'},
    'Wrack': {'color': 'purple',  'icon': 'caret-up'},
    'Maerl': {'color': 'pink', 'icon': 'circle'},
    'CCA':   {'color': 'cadetblue', 'icon': 'star'}
}

marker_cluster = MarkerCluster(spiderfyOnMaxZoom=True).add_to(m)

# 3. Use the SNAPPED coordinates for the markers
for group, style in shape_map.items():
    # Filter using your final algae_snap dataframe
    subset = algae_snap[algae_snap['Functional Group'].str.contains(group, na=False)]

    for _, row in subset.iterrows():
        # IMPORTANT: Use the new snap_lat/snap_lon we generated
        lat, lon = row['snap_lat'], row['snap_lon']

        folium.Marker(
            location=[lat, lon],
            icon=folium.Icon(color=style['color'], icon=style['icon'], prefix='fa'),
            tooltip=(f"<b>Grouped Site:</b> {row['Standard_Site_Name']}<br>"
                     f"<b>Group:</b> {group}<br>"
                     f"<b>Order:</b> {row['Order']}<br>"
                     f"<b>Species:<b> {row['TaxonName']}<br>"
                     f"<b>Original Site:</b> {row['SiteName']}<br>"
                     f"<b>Lat:</b> {lat:.5f}<br>"
                     f"<b>Lon:</b> {lon:.5f}<br>"
                     f"<b>Date:<b> {row['Date']}<br>"
                     f"<b>Source:<b> {row['Original_Data']}")
        ).add_to(marker_cluster)


# 4. Add a layer control so you can turn Kelp/Maerl on and off
folium.LayerControl().add_to(m)

# Save to an HTML file you can open in any browser
m.save('interactive_algae_map.html')
m

In [ ]:
# 1. Create a base map centered on your Connemara coordinates
# We use 'cartodbpositron' for a clean, gray background similar to your static map
m = folium.Map(location=[53.35, -9.85], zoom_start=10, tiles='cartodbpositron')

# 2. Define your styles to match your previous plot
shape_map = {
    'Kelp':  {'color': 'green',   'icon': 'square'},
    'Wrack': {'color': 'purple',  'icon': 'caret-up'},
    'Maerl': {'color': 'pink', 'icon': 'circle'},
    'CCA':   {'color': 'cadetblue', 'icon': 'star'}
}

marker_cluster = MarkerCluster(spiderfyOnMaxZoom=True).add_to(m)

# 3. Use the SNAPPED coordinates for the markers
for group, style in shape_map.items():
    # Filter using your final algae_snap dataframe
    subset = algae_snap[algae_snap['Functional Group'].str.contains(group, na=False)]

    for _, row in subset.iterrows():
        # IMPORTANT: Use the new snap_lat/snap_lon we generated
        lat, lon = row['snap_lat'], row['snap_lon']

        folium.Marker(
            location=[lat, lon],
            icon=folium.Icon(color=style['color'], icon=style['icon'], prefix='fa'),
            tooltip=(f"<b>Grouped Site:</b> {row['Standard_Site_Name']}<br>"
                     f"<b>Group:</b> {group}<br>"
                     f"<b>Order:</b> {row['Order']}<br>"
                     f"<b>Species:<b> {row['TaxonName']}<br>"
                     f"<b>Original Site:</b> {row['SiteName']}<br>"
                     f"<b>Lat:</b> {lat:.5f}<br>"
                     f"<b>Lon:</b> {lon:.5f}<br>"
                     f"<b>Date:<b> {row['Date']}<br>"
                     f"<b>Source:<b> {row['Original_Data']}")
        ).add_to(marker_cluster)


# 4. Add a layer control so you can turn Kelp/Maerl on and off
folium.LayerControl().add_to(m)

# Save to an HTML file you can open in any browser
m.save('interactive_algaecommunity_map.html')
m
## Im not sure how this is any different than 'interactive_algae_map.html'

In [ ]:
algae_snap['Functional Group'].unique()

In [ ]:
algae_snap['Year'] = algae_snap['Date'].dt.year.astype(str)

In [ ]:
import folium
import folium.plugins as plugins

# 1. Setup the map
m = folium.Map(location=[53.35, -9.85], zoom_start=10, tiles='cartodbpositron')

# 2. Create the Parent Cluster (The "Master")
# This handles the high-level grouping
parent_cluster = plugins.MarkerCluster(spiderfyOnMaxZoom=True).add_to(m)

# 3. Define styles (same as before)
shape_map = {
    'Kelp':                  {'color': 'darkgreen', 'icon': 'leaf'},
    'Wrack':                 {'color': 'orange',    'icon': 'chevron-up'},
    'Subtidal Brown':        {'color': 'beige',     'icon': 'anchor'},
    'Maerl':                 {'color': 'lightred',  'icon': 'certificate'},
    'CCA':                   {'color': 'cadetblue', 'icon': 'star'},
    'Geniculate Coralline':  {'color': 'pink',      'icon': 'plus'},
    'Fleshy Red':            {'color': 'purple',    'icon': 'tint'},
    'Fine Red':              {'color': 'darkpurple','icon': 'ellipsis-h'},
    'Other Green':           {'color': 'green',     'icon': 'envira'},
    'Opportunistic':         {'color': 'red',       'icon': 'exclamation-triangle'}
}

# 4. Create Subgroups using the class you found in the docs
subgroups = {}
unique_years = sorted(algae_snap['Year'].unique())

for year in unique_years:
    # This is the magic step: it creates independent year-bins within the cluster
    subgroups[year] = plugins.FeatureGroupSubGroup(
        parent_cluster,
        name=f"Year {year}"
    ).add_to(m)

# 5. Populate the map
for group, style in shape_map.items():
    subset = algae_snap[algae_snap['Functional Group'].str.contains(group, na=False)]

    for _, row in subset.iterrows():
        year = row['Year']
        lat, lon = row['snap_lat'], row['snap_lon']

        folium.Marker(
            location=[lat, lon],
            icon=folium.Icon(color=style['color'], icon=style['icon'], prefix='fa'),
            tooltip=(f"<b>Year:</b> {year}<br>"
                     f"<b>Group:</b> {group}<br>"
                     f"<b>Species:</b> {row['TaxonName']}")
        ).add_to(subgroups[year])

# 6. Finalize
folium.LayerControl(collapsed=True).add_to(m)
m.save('subgroup_community_map.html')
m

## has all years and all subgroups in one massive web, its too busy and hard to read

In [ ]:
import folium
from folium.plugins import MarkerCluster

# 1. Setup the map
m = folium.Map(location=[53.35, -9.85], zoom_start=11, tiles='cartodbpositron')

# 2. Create the Cluster - we reduce the 'maxClusterRadius'
# This forces the clusters to split up much sooner when you zoom in
main_cluster = MarkerCluster(
    spiderfyOnMaxZoom=True,
    maxClusterRadius=30  # Smaller radius = more sensitive to the nudges
).add_to(m)

# 3. Styles (10 groups)
shape_map = {
    'Kelp':                  {'color': 'darkgreen', 'icon': 'leaf'},
    'Wrack':                 {'color': 'orange',    'icon': 'chevron-up'},
    'Subtidal Brown':        {'color': 'beige',     'icon': 'anchor'},
    'Maerl':                 {'color': 'lightred',  'icon': 'certificate'},
    'CCA':                   {'color': 'cadetblue', 'icon': 'star'},
    'Geniculate Coralline':  {'color': 'pink',      'icon': 'plus'},
    'Fleshy Red':            {'color': 'purple',    'icon': 'tint'},
    'Fine Red':              {'color': 'darkpurple','icon': 'ellipsis-h'},
    'Other Green':           {'color': 'green',     'icon': 'envira'},
    'Opportunistic':         {'color': 'red',       'icon': 'exclamation-triangle'}
}

# 4. Apply the High-Contrast Nudge
# 2024 is our "Anchor" (Center).
# 2007 will be significantly South, 2026 will be slightly North.
for _, row in algae_snap.iterrows():
    year = int(row['Year'])

    # Significant Nudge: approx 15-18m per step
    # This ensures 2017/2018/2020 are far enough apart to NOT group
    lat_nudge = (year - 2024) * 0.00015
    lat, lon = row['snap_lat'] + lat_nudge, row['snap_lon']

    # Match functional group
    group = next((g for g in shape_map if g in row['Functional Group']), 'Opportunistic')
    style = shape_map[group]

    folium.Marker(
        location=[lat, lon],
        icon=folium.Icon(color=style['color'], icon=style['icon'], prefix='fa'),
        tooltip=(f"<b>Year:</b> {year}<br>"
                 f"<b>Site:</b> {row['Standard_Site_Name']}<br>"
                 f"<b>Group:</b> {group}<br>"
                 f"<b>Taxon:</b> {row['TaxonName']}")
    ).add_to(main_cluster)


m.save('year_grouped_spiderwebs.html')
m

## for this cell, want to make sure that within the yearly groups, that they are still presented in order of functional group. ie. I don't want the kelps to be spread out in the web, they should be next to each other
## but overall this is closer to what I am looking for than anything else
## would still want to get it into a form why they are grouped by year, but still attached to the same focal point, then break off into three different spirals

## Jaccard Index

From IBM: "Jaccard similarity is a statistical measure used to quantify how similar two sets are. It equals the ratio of the intersection size to the union size. The value of Jaccard similarity ranges from zero to one, where **zero indicates that the sets have no elements in common** and **one indicates that the sets are identical**"

https://www.ibm.com/think/topics/jaccard-similarity#:~:text=Jaccard%20similarity%20is%20a%20statistical,that%20the%20sets%20are%20identical.

Reasons why I think it is relevant:
"The scientist Paul Jaccard developed Jaccard similarity at the beginning of the 20th century to help differentiate species of plants." - developed with environmental science in mind.
"Jaccard similarity is widely used when data is represented as sets or as binary features indicating the presence or absence." - my data is presence/absense.


In [ ]:
## need to add effort level so that it is easier to determine sampling effort bias

In [ ]:
def jaccard(df):

    # agg species into set per location per year
    df = (df.groupby(['snap_lat','snap_lon', 'Year']).agg({
        'TaxonName': lambda x: set(x),
        'Date': 'nunique'
    }).reset_index())

    df.rename(columns={'Date':'Survey_Count'}, inplace=True)

    results = []

    # group by location
    for (lat,lon), group in df.groupby(['snap_lat','snap_lon']):
        group = group.sort_values('Year')
        years = group['Year'].tolist()
        species_sets = group['TaxonName'].tolist()
        effort = group['Survey_Count'].tolist()

        ## need two years, if code is false (greater than 1 year), then it works through the rest of the code)
        if len(years) < 2:
            continue

        for i in range(len(years) -1):
            y1, y2 = years[i], years[i+1]
            set1, set2 = species_sets[i], species_sets[i+1] #species
            eff1, eff2 = effort[i], effort[i+1] #effort
            rich1, rich2 = len(set1), len(set2) ##richness

            appeared = set2 - set1
            dissappeared = set1 - set2
            union = set1 | set2

            ##Jaccard similarity
            similarity = len(set1 & set2) / len(union) if union else 1.0

            # Richness Ratio (Year2/Year1) Find outliers
            rich_ratio = rich2/rich1 if rich1>0 else 0

            results.append({
                'Latitude': lat,
                'Longitude': lon,
                'Start_Year': y1,
                'Effort_Start_Year': eff1,
                'Rich_Start_Year': rich1,
                'End_Year':y2,
                'Effort_End_Year': eff2,
                'Rich_End_Year': rich2,
                'Year_Interval': f"{y1}-{y2}",
                'Effort_Diff': eff2 - eff1, ## negative means less effort in the second year
                'Richness_Ratio': round(rich_ratio,4),
                'Appeared': list(appeared),
                'Disappeared': list(dissappeared),
                'n_Appeared': len(appeared),
                'n_Disappeared': len(dissappeared),
                'Jaccard_Index': round(similarity, 4),
                'Total_Spec_Involved': len(union)})

    # geodataframe
    results_df = pd.DataFrame(results)
    results_gdf = gpd.GeoDataFrame(results_df, geometry=gpd.points_from_xy(results_df.Longitude, results_df.Latitude), crs = "EPSG:4326")

    return results_gdf



In [ ]:
test_df = algae_snap[np.isclose(algae_snap['snap_lat'], 51.501524) & np.isclose(algae_snap['snap_lon'], -9.427154)]
test_df
#51.501524,-9.427154


In [ ]:
test_jaccard = jaccard(test_df)

In [ ]:
test_jaccard

In [ ]:
jaccard_algae = jaccard(algae_snap)

In [ ]:
jaccard_algae.sort_values('Effort_Diff')

In [ ]:
print(
    "The mean Jaccard Index with -5 Effort Difference is: ", jaccard_algae[jaccard_algae['Effort_Diff'] == -5]['Jaccard_Index'].mean(), "\n",
    "The mean Jaccard Index with -3 Effort Difference is: ", jaccard_algae[jaccard_algae['Effort_Diff'] == -3]['Jaccard_Index'].mean(), "\n",
    "The mean Jaccard Index with -2 Effort Difference is: ", jaccard_algae[jaccard_algae['Effort_Diff'] == -2]['Jaccard_Index'].mean(), "\n",
    "The mean Jaccard Index with -1 Effort Difference is: ", jaccard_algae[jaccard_algae['Effort_Diff'] == -1]['Jaccard_Index'].mean(), "\n",
    "The mean Jaccard Index with 0 Effort Difference is: ", jaccard_algae[jaccard_algae['Effort_Diff'] == 0]['Jaccard_Index'].mean(), "\n",
    "The mean Jaccard Index with +1 Effort Difference is: ", jaccard_algae[jaccard_algae['Effort_Diff'] == 1]['Jaccard_Index'].mean())

In [ ]:
def fivenumsum(v):

    q1 = np.percentile(v, 25)
    q3 = np.percentile(v, 75)
    median = np.median(v)
    minimum = np.min(v)
    maximum = np.max(v)

    stats = [minimum, q1, median, q3, maximum]

    return ", ".join([f"{x:.3f}" for x in stats])


In [ ]:
print(
    "The 5 number summary of Jaccard Index with -5 Effort Difference is: ", fivenumsum(jaccard_algae[jaccard_algae['Effort_Diff'] == -5]['Jaccard_Index']), "\n",
    "The 5 number summary of Jaccard Index with -3 Effort Difference is: ", fivenumsum(jaccard_algae[jaccard_algae['Effort_Diff'] == -3]['Jaccard_Index']), "\n",
    "The 5 number summary of Jaccard Index with -2 Effort Difference is: ", fivenumsum(jaccard_algae[jaccard_algae['Effort_Diff'] == -2]['Jaccard_Index']), "\n",
    "The 5 number summary of Jaccard Index with -1 Effort Difference is: ", fivenumsum(jaccard_algae[jaccard_algae['Effort_Diff'] == -1]['Jaccard_Index']), "\n",
    "The 5 number summary of Jaccard Index with 0 Effort Difference is: ", fivenumsum(jaccard_algae[jaccard_algae['Effort_Diff'] == 0]['Jaccard_Index']), "\n",
    "The 5 number summary of Jaccard Index with +1 Effort Difference is: ", fivenumsum(jaccard_algae[jaccard_algae['Effort_Diff'] == 1]['Jaccard_Index']))

In [ ]:
## easier way
effort_stats = jaccard_algae.groupby('Effort_Diff')['Jaccard_Index'].describe()
effort_stats.iloc[3:,:]

In [ ]:
## can see here that the median for effort_diff = -5 is significantly lower than effort_diff = 0
## i'm not going to include the -5, -3, -2, -1 or +1, I'm only going to include the sites that have an equal amount of effort while sampling

In [ ]:
## richness ration : aiming for values greater than or equal to 0.5 (half richness), or less than or equal to 2.0 (double richness)

In [ ]:
jaccard_algae_clean = jaccard_algae[(jaccard_algae['Effort_Diff'] == 0) & (jaccard_algae['Richness_Ratio'] >= 0.5) & (jaccard_algae['Richness_Ratio'] <= 2.0)]
jaccard_algae_clean

In [ ]:
jaccard_algae_clean = jaccard_algae_clean.to_crs('EPSG:2157')

In [ ]:
jaccard_algae_clean.sort_values('End_Year', inplace = True)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))

island.plot(ax=ax, color="lightgray", edgecolor="black")

jaccard_algae_clean.plot(
    ax=ax,
    column='Jaccard_Index',
    cmap='viridis',
    markersize=40,
    legend = True,
    edgecolor='white',
    linewidth=0.5,
    legend_kwds={'label': "Jaccard Similarity Index", 'orientation': "horizontal", 'pad': 0.05}
)


plt.title('Stability of Irish Macroalgae Communities (Jaccard Index)')
plt.show()

### problems:
1. jaccard only tells me that it is changing, not for the better or for the worse
- Need to do a Net Change columns, with n_Appeared-n_Dissapeared
2. a lot less data than I started with (upon review, not that much but only 5 sites in galway bay with an interval)
3. Still need to do site by site comparisons between the different buckets of years
4. Still need to relate this to coastal darkening satellite data AND to riverine discharge data (still need to finish processing the riverine data)

In [ ]:
jaccard_df = jaccard_algae_clean
jaccard_df

In [ ]:
jaccard_df['Net_Change'] = jaccard_df['n_Appeared'] - jaccard_df['n_Disappeared']

In [ ]:
jaccard_df

In [ ]:
from pyproj import CRS, Proj, transform
crs_itm = CRS.from_epsg(2157)
crs_itm

In [ ]:
crs_wgs = CRS.from_epsg('4326')
crs_wgs

In [ ]:
jaccard_df['Easting'] = jaccard_df.geometry.x
jaccard_df['Northing'] = jaccard_df.geometry.y
jaccard_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))

island.plot(ax=ax, color="lightgray", edgecolor="black")

jaccard_df.plot(
    ax=ax,
    column='Jaccard_Index',
    cmap='viridis',
    markersize=40,
    legend = True,
    edgecolor='white',
    linewidth=0.5,
    legend_kwds={'label': "Jaccard Similarity Index", 'orientation': "horizontal", 'pad': 0.05}
)

plt.xlim(470000,550000)
plt.ylim(700000,735000)

plt.title('Stability of Irish Macroalgae Communities (Jaccard Index)')
plt.show()

In [ ]:
jaccard_df[(jaccard_df.geometry.y < 710000) & (jaccard_df.geometry.y > 705000) & (jaccard_df.geometry.x < 520000) & (jaccard_df.geometry.x > 510000)]

## this appears to be an outlier

Can see that the point in question (above) has a very low jaccard, but that is because of surveying bias (most likely). To elimate sites such as this, a minimum richness level of >= 10 species surveys per site is added below

In [ ]:
jaccard_df['Rich_Start_Year'].describe()

In [ ]:
jaccard_df = jaccard_df[(jaccard_df.Rich_Start_Year >=10) & (jaccard_df.Rich_End_Year >=10)]
jaccard_df
## elimated 25 site time buckets

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))

island.plot(ax=ax, color="lightgray", edgecolor="black")

jaccard_df.plot(
    ax=ax,
    column='Jaccard_Index',
    cmap='viridis',
    markersize=40,
    legend = True,
    edgecolor='white',
    linewidth=0.5,
    legend_kwds={'label': "Jaccard Similarity Index", 'orientation': "horizontal", 'pad': 0.05}
)

plt.xlim(470000,550000)
plt.ylim(700000,735000)

plt.title('Stability of Irish Macroalgae Communities (Jaccard Index)')
plt.show()

In [ ]:
## use this cell to identify coordinates of sites
jaccard_df[(jaccard_df.geometry.y < 725000) & (jaccard_df.geometry.y > 720000) & (jaccard_df.geometry.x < 500000) & (jaccard_df.geometry.x > 490000)]

#### New task: combine algae_snap and jaccard_df, first goal is to get the site names in there

In [ ]:
algae_snap


In [ ]:
names_loc = algae_snap[['Standard_Site_Name', 'snap_lat','snap_lon']]
names_loc = names_loc.drop_duplicates()
names_loc.rename(columns={'snap_lat':'Latitude', 'snap_lon':'Longitude'}, inplace=True)
names_loc

In [ ]:
## So i want to keep all the lat lon pairs that are in jaccard_df, and match them to their corresponding lat lon pairs in names_loc and have them add the standard site name

attempt_1 = pd.merge(jaccard_df, names_loc, how='left', on=['Latitude', 'Longitude'])
attempt_1

In [ ]:
## yay it worked, now assign copy of attempt 1 to jaccard
jaccard_df = attempt_1.copy()

In [ ]:
jaccard_df.dtypes

In [ ]:
jaccard_df = jaccard_df.astype({'Start_Year':int, 'End_Year':int})

In [ ]:
jaccard_df.dtypes

In [ ]:
def plot_site_timeline_by_coords(df, latitude, longitude):
    ## filter to the coordinates
    site_data = df[
        (df['Latitude'].between(latitude - 0.0001, latitude + 0.0001)) &
        (df['Longitude'].between(longitude - 0.0001, longitude + 0.0001))
    ].copy()

    ## use midpoint to represent intervals
    site_data['Interval_Span'] = site_data['End_Year'] - site_data['Start_Year']
    site_data['Midpoint'] = site_data['Start_Year'] + (site_data['Interval_Span']/2)
    site_data = site_data.sort_values('Midpoint')

    ## if the lat/lon isn't good, it doesn't just show an empty graph but tells you what is wrong
    if site_data.empty:
        print("No data found.")
        return

    fig, ax1 = plt.subplots(figsize=(10, 6))

    ## Axis 1 - Jaccard
    ax1.plot(site_data['Midpoint'], site_data['Jaccard_Index'], color='black', marker='o', markersize= 8, linewidth=1.5, label = "Jaccard Index")
    ax1.set_ylabel('Jaccard Similarity Index', fontsize = 12, fontweight='bold')
    ax1.grid(True, alpha= 0.3, linestyle = '--', zorder = 0)

    ## Axis 2 - Net Change
    ax2 = ax1.twinx()

    ## green for gain, red for loss (6 is green, d is red, and 3 is indigo
    colors = ['#60e69f' if x >0 else '#d12e54' if x < 0 else '#362ed1' for x in site_data['Net_Change']]

    ax2.bar(site_data['Midpoint'], site_data['Net_Change'],
            color=colors, alpha = 0.5, edgecolor='white',
            width = site_data['Interval_Span'], ## make the width of the bar proportional to the gap in time between years in interval
            label = 'Net Species Change')

    ## add a label for intervals where net change is zero
    for i, row in site_data.iterrows():
        if row['Net_Change'] == 0:
            ax2.hlines(0, row['Start_Year'], row['End_Year'],
                       colors='#362ed1', linewidth=3, alpha = 0.5)



    ax2.set_ylabel('Net Species Change (Count)', fontsize = 12, fontweight='bold')

    #center the 0 line
    limit = max(abs(site_data['Net_Change']).max() +5, 20)
    ax2.set_ylim(-limit, limit)
    ax2.axhline(0, color='black', linestyle='--', linewidth=1, alpha = 0.5)
    ax2.grid(True, alpha= 0.3, linestyle = '--', zorder = 1)
    ## want x axis labels to show interval range
    ax1.set_xticks(site_data['Midpoint'])
    tick_labels = [f"{int(s)}-{int(e)}" for s, e in zip(site_data['Start_Year'], site_data['End_Year'])]
    ax1.set_xticklabels(tick_labels)

    ax1.set_title(f'Temporal Community Dynamics:\n{site_data['Standard_Site_Name'].iloc[0]}', fontsize = 14, fontweight='bold', pad=25)

    ax2.set_title(f'Location: {latitude}, {longitude}', fontsize = 11, pad = 10)

    ax1.set_xlabel('Survey Year (End of Interval)', fontsize = 12)

    ## legendddd
    legend_elements = [
        plt.Line2D([0], [0], color='black', marker='o', markersize=8, label='Jaccard Index'),
        Patch(facecolor='#60e69f', alpha=0.5, label='Species Gain (Net)'),
        Patch(facecolor='#d12e54', alpha=0.5, label='Species Loss (Net)'),
        Patch(facecolor='#362ed1', alpha=0.5, label='No Net Change')
    ]
    leg = ax1.legend(handles = legend_elements, loc= 'upper right', bbox_to_anchor = (1.3,1), fontsize=10, frameon = True)


    plt.tight_layout()
    plt.show()

In [ ]:

plot_site_timeline_by_coords(jaccard_df, 53.245957,-9.628365)


In [ ]:
jaccard_df[(jaccard_df.geometry.y < 730000) & (jaccard_df.geometry.y > 725000) & (jaccard_df.geometry.x < 490000) & (jaccard_df.geometry.x > 480000)]

In [ ]:

plot_site_timeline_by_coords(jaccard_df, 53.294096,-9.771043)


In [ ]:
## sites of particular interest will have multiple, multi year surveys (im going crazy)